# Neural Networks in PySpark

Who has not had that dream? In the middle of the night, we lie awake and think of PySpark. Then, in the silvery moonshine, we think of neural networks and an ardent wish fills our hearts: how can we run deep learning models on Spark?

This notebook covers three things:
1. **PyTorch on Spark** — train a small CNN on MNIST using ``TorchDistributor``, Spark's mechanism for launching PyTorch jobs from a Spark session.
2. **Transfer learning** — use a pre-trained ResNet-50 to extract image features inside Spark via ``predict_batch_udf``, then classify them with both Spark ML and a PyTorch head trained through ``TorchDistributor``.
3. **Large-scale inference** — apply a pre-trained DistilBERT sentiment model to every line of Shakespeare, then use Spark SQL to analyse sentiment per play and per character.

### Setup

Install dependencies if running locally:

In [1]:
!pip install torch torchvision pillow transformers pyarrow

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Deep_Learning")
    .config("spark.driver.memory", "12g")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/22 19:31:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Shared utilities

``download_with_progress`` is used for the flower dataset.

In [3]:
import urllib.request
import tarfile
from pathlib import Path

def download_with_progress(url: str, dest: Path) -> None:
    def _progress(count, block_size, total_size):
        pct = count * block_size / total_size * 100
        print(f"\r  Downloading... {min(pct, 100):.1f}%", end="", flush=True)
    print(f"Downloading from {url}")
    urllib.request.urlretrieve(url, dest, reporthook=_progress)
    print(f"\n  Saved to {dest} ({dest.stat().st_size / 1e6:.1f} MB)")

## PyTorch on Spark: MNIST

Before working with images and language, we introduce ``TorchDistributor`` on the simplest possible task: classifying handwritten digits.

The pattern is always the same:
1. Write a **self-contained training function** — all imports inside, no driver state.
2. Hand it to ``TorchDistributor.run()``.
3. Receive the trained ``state_dict`` back on the driver.

Everything else in the notebook follows this pattern.

In [2]:
from pyspark.ml.torch.distributor import TorchDistributor
import torch
import torch.nn as nn
import torch.nn.functional as F
import pyarrow.parquet as pq
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset, random_split


### The network

A small CNN: two convolutional layers to extract spatial features from the 28×28 images, max-pooling to downsample, then two linear layers down to 10 output logits — one per digit class. The forward pass returns raw logits; ``F.cross_entropy`` handles the softmax.

In [5]:
# Tensor shapes throughout: B = batch size, C = channels, H = height, W = width
# e.g. [B, 1, 28, 28] means a batch of B grayscale images, each 28×28 pixels.

class MNISTNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3),   # [B,  1, 28, 28] → [B, 32, 26, 26]  (32 filters, kernel 3×3)
            nn.ReLU(),
            nn.MaxPool2d(2),       # [B, 32, 26, 26] → [B, 32, 13, 13]  (halve spatial dims)
            nn.Conv2d(32, 64, 3),  # [B, 32, 13, 13] → [B, 64, 11, 11]
            nn.ReLU(),
            nn.Flatten(),          # [B, 64, 11, 11] → [B, 7744]         (64×11×11 = 7744)
            nn.Linear(7744, 128),
            nn.ReLU(),
            nn.Linear(128, 10),    # [B, 10] — one logit per digit class
        )

    def forward(self, x):
        return self.net(x)

### Let's code!

Fill in ``train_mnist`` below and run it via ``TorchDistributor``.

The function must be **fully self-contained** — all imports must be repeated inside the function body, because ``TorchDistributor`` pickles it and sends it to a separate worker process that does not share the driver's namespace.

Steps:
1. Repeat imports inside the function.
2. Download MNIST with ``datasets.MNIST`` and wrap in ``DataLoader``s.
3. Write a training loop for ``EPOCHS`` epochs using ``F.cross_entropy``.
4. Return ``model.state_dict()``.

See the ``TorchDistributor`` documentation for more details: https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.torch.distributor.TorchDistributor.html

In [6]:
EPOCHS     = 3
BATCH_SIZE = 64
LR         = 1e-3
DATA_PATH  = "/tmp/mnist"


def train_mnist():
    # All imports and class definitions must live inside the function:
    # TorchDistributor pickles it and sends it to a worker process that
    # does not share the driver's module namespace.
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torch.optim as optim
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader, random_split

    transform    = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),  # MNIST channel mean and std
    ])
    full_train   = datasets.MNIST(DATA_PATH, train=True,  download=True, transform=transform)
    test_ds      = datasets.MNIST(DATA_PATH, train=False, download=True, transform=transform)
    train_ds, val_ds = random_split(full_train, [0.8, 0.2])

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model  = MNISTNet().to(device)
    opt    = optim.Adam(model.parameters(), lr=LR)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            opt.zero_grad()
            F.cross_entropy(model(images), labels).backward()
            opt.step()

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                correct += (model(images).argmax(1) == labels).sum().item()
                total   += labels.size(0)
        print(f"Epoch {epoch}/{EPOCHS}  val_acc={correct/total:.4f}")

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            correct += (model(images).argmax(1) == labels).sum().item()
            total   += labels.size(0)
    print(f"Test accuracy: {correct/total:.4f}")

    return model.state_dict()


state_dict = TorchDistributor(
    num_processes=2, local_mode=True, use_gpu=False
).run(train_mnist)

# Reconstruct the model on the driver to enable local inference
trained_model = MNISTNet()
trained_model.load_state_dict(state_dict)
trained_model.eval()


Started local training with 2 processes



*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed.
*****************************************
100%|██████████| 9.91M/9.91M [00:00<00:00, 36.3MB/s]
100%|██████████| 9.91M/9.91M [00:00<00:00, 35.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.06MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.05MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.7MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.20MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 15.5MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 17.0MB/s]
Epoch 1/3  val_acc=0.9824
Epoch 1/3  val_acc=0.9782
Epoch 2/3  val_acc=0.9841
Epoch 2/3  val_acc=0.9864
Epoch 3/3  val_acc=0.9840
Test accuracy: 0.9866
Epoch 3/3  val_acc=0.9878
Test accuracy: 0.9892


Finished local training with 2 processes


MNISTNet(
  (net): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
    (4): ReLU()
    (5): Flatten(start_dim=1, end_dim=-1)
    (6): Linear(in_features=7744, out_features=128, bias=True)
    (7): ReLU()
    (8): Linear(in_features=128, out_features=10, bias=True)
  )
)

<!-- ## Transfer Learning with Spark

A very common production pattern is to run a **pre-trained model** across a large dataset to extract features, then feed those features into a lighter Spark ML model. This is called *transfer learning* — we borrow the visual understanding a large model has already learned and redirect it to our own task.

Spark 3.4 introduced ``predict_batch_udf`` (``pyspark.ml.functions``) as the idiomatic way to run distributed inference. Compared to a hand-written Pandas UDF:
- the model is loaded **once per worker process** at startup and reused across all batches that worker handles — no repeated loading,
- batching is managed by the API — you just specify ``batch_size``,
- no manual broadcasting or pickling of the model object is needed.

We will use the **flower photos dataset** (3,670 images, 5 classes) and extract 2048-dimensional feature vectors using a pre-trained **ResNet-50**. -->

### Downloading the flower dataset

We use the TensorFlow flower photos dataset — 3,670 images across 5 classes (daisy, dandelion, roses, sunflowers, tulips). The helper below downloads and extracts the archive with a progress indicator.

In [19]:
URL      = "http://download.tensorflow.org/example_images/flower_photos.tgz"
DEST_DIR = Path("flower_photos")
ARCHIVE  = Path("flower_photos.tgz")

if not DEST_DIR.exists():
    if not ARCHIVE.exists():
        download_with_progress(URL, ARCHIVE)
    with tarfile.open(ARCHIVE) as tar:
        tar.extractall()
else:
    print(f"{DEST_DIR}/ already exists, skipping download.")

flower_photos/ already exists, skipping download.


### Loading images into Spark

Spark's ``binaryFile`` format reads image files as raw bytes into a DataFrame. Each row contains the file path and the raw content. We extract the flower class from the directory name (the second component of the path).

In [8]:
from pyspark.sql.functions import col, split, element_at

images = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.jpg")
    .option("recursiveFileLookup", "true")
    .load(str(DEST_DIR))
    .sample(fraction=0.1, seed=753)
)

# Path looks like: .../flower_photos/roses/123.jpg
# element_at with -2 picks the second-to-last segment (the class directory),
# which is robust regardless of how many leading path components there are.
images = images.withColumn(
    "label",
    element_at(split(col("path"), "/"), -2)
)

images.show(5, truncate=True)


+--------------------+-------------------+------+--------------------+----------+
|                path|   modificationTime|length|             content|     label|
+--------------------+-------------------+------+--------------------+----------+
|file:/teamspace/s...|2016-01-11 06:54:55|281953|[FF D8 FF E0 00 1...|    tulips|
|file:/teamspace/s...|2016-01-11 06:19:49|235856|[FF D8 FF E0 00 1...|sunflowers|
|file:/teamspace/s...|2016-01-11 06:10:31|234384|[FF D8 FF E0 00 1...| dandelion|
|file:/teamspace/s...|2016-01-11 06:14:39|228886|[FF D8 FF E0 00 1...|     roses|
|file:/teamspace/s...|2016-01-11 06:57:47|224926|[FF D8 FF E0 00 1...|    tulips|
+--------------------+-------------------+------+--------------------+----------+
only showing top 5 rows


### Extracting features with ResNet-50

We use **ResNet-50** with its default pretrained weights and remove the final classification head, so the model outputs a 2048-dimensional feature vector per image.

``predict_batch_udf`` takes a **setup function** — here ``make_resnet_fn`` — that each worker calls once at startup to load the model. The inner ``predict`` function it returns is then called repeatedly on batches of images.

In [9]:
from pyspark.ml.functions import predict_batch_udf
from pyspark.sql.types import FloatType, ArrayType


def make_resnet_fn():
    """Called once per worker at startup: loads and caches the model."""
    import io, torch, numpy as np
    from PIL import Image
    from torchvision.models import resnet50, ResNet50_Weights
    from torchvision import transforms

    model = resnet50(weights=ResNet50_Weights.DEFAULT)
    model = torch.nn.Sequential(*list(model.children())[:-1])  # drop classifier head
    model.eval()

    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],   # RGB channel means of ImageNet
                             std=[0.229, 0.224, 0.225]),    # RGB channel stds  of ImageNet
    ])

    def predict(image_bytes_batch: np.ndarray) -> np.ndarray:
        tensors = [
            preprocess(Image.open(io.BytesIO(b)).convert("RGB"))
            for b in image_bytes_batch
        ]
        with torch.no_grad():
            return model(torch.stack(tensors)).squeeze(-1).squeeze(-1).numpy()

    return predict


extract_features_udf = predict_batch_udf(
    make_resnet_fn,
    return_type=ArrayType(FloatType()),
    batch_size=32,
)

We repartition to control parallelism, then apply the UDF. Each partition is processed by one Python worker, which loads ResNet-50 once and runs it over its assigned images in batches of 32.

In [10]:
features_df = (
    images
    .repartition(8)
    .select(
        col("path"),
        col("label"),
        extract_features_udf(col("content")).alias("features")
    )
    .cache()
)

features_df.show(5, truncate=True)

+--------------------+----------+--------------------+
|                path|     label|            features|
+--------------------+----------+--------------------+
|file:/teamspace/s...|sunflowers|[0.06216928, 0.0,...|
|file:/teamspace/s...|sunflowers|[0.058896415, 0.0...|
|file:/teamspace/s...|sunflowers|[0.00408922, 0.02...|
|file:/teamspace/s...| dandelion|[0.15540065, 0.0,...|
|file:/teamspace/s...|sunflowers|[0.0, 0.0, 0.2793...|
+--------------------+----------+--------------------+
only showing top 5 rows


### Exercise

Adapt the code below to run a multi-class flower classification using the ResNet features.

The features are stored as an ``ArrayType(FloatType())`` column. Use ``array_to_vector`` to convert them to the ``DenseVector`` format that Spark ML expects, then split into train/test and fit a classifier.

A ``LogisticRegression`` baseline is provided. Try swapping in a ``RandomForestClassifier`` or ``GradientBoostedTreesClassifier`` and compare accuracy.

In [11]:
from pyspark.ml.functions import array_to_vector
from pyspark.ml.feature import StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

indexer           = StringIndexer(inputCol="label", outputCol="labelIndex")
features_df_label = indexer.fit(features_df).transform(features_df)

features_df_dense = features_df_label.withColumn(
    "features_vectorized", array_to_vector(col("features"))
)

train, test = features_df_dense.randomSplit([0.7, 0.3], seed=7531453)

features_df_dense.show(5, truncate=True)

+--------------------+----------+--------------------+----------+--------------------+
|                path|     label|            features|labelIndex| features_vectorized|
+--------------------+----------+--------------------+----------+--------------------+
|file:/teamspace/s...|sunflowers|[0.06216928, 0.0,...|       3.0|[0.06216927990317...|
|file:/teamspace/s...|sunflowers|[0.058896415, 0.0...|       3.0|[0.05889641493558...|
|file:/teamspace/s...|sunflowers|[0.00408922, 0.02...|       3.0|[0.00408921996131...|
|file:/teamspace/s...| dandelion|[0.15540065, 0.0,...|       1.0|[0.15540064871311...|
|file:/teamspace/s...|sunflowers|[0.0, 0.0, 0.2793...|       3.0|[0.0,0.0,0.279328...|
+--------------------+----------+--------------------+----------+--------------------+
only showing top 5 rows


Logistic regression in PySpark supports both binary and multinomial classification. For multinomial, the model estimates one probability per class and predicts the class with the highest score.

We add light L2 regularisation (``regParam``) to avoid overfitting on the small sample.

In [12]:
lr = LogisticRegression(
    featuresCol="features_vectorized",
    labelCol="labelIndex",
    regParam=0.1,
    elasticNetParam=0.0,  # 0 = pure L2
    maxIter=1000
)
logistic_model = lr.fit(train)

26/04/22 09:07:48 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


In [13]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="labelIndex",
    predictionCol="prediction",
    metricName="accuracy"
)
test_pred = logistic_model.transform(test)
evaluator.evaluate(test_pred)

0.822429906542056

### Training a classification head with TorchDistributor

The logistic regression baseline treats Spark ML as a black box. We can do better — and connect both halves of the notebook — by training a small PyTorch **classification head** on the same ResNet features using ``TorchDistributor``.

The flow is the same as in the Shakespeare section:
1. Spark writes the features to Parquet — the handoff point.
2. ``TorchDistributor`` launches PyTorch, which reads the Parquet files.
3. The trained weights come back to the driver as a ``state_dict``.

The model is deliberately simple: two fully-connected layers with ReLU and dropout. The ResNet has already done the hard visual work; the head only needs to learn which regions of the 2048-dimensional feature space correspond to each flower class.

In [14]:
FLOWERS_TRAIN = "flowers_train.parquet"
FLOWERS_TEST  = "flowers_test.parquet"

# Persist the labelled feature vectors so TorchDistributor can read them
(
    train
    .select("features_vectorized", "labelIndex")
    .write.mode("overwrite").parquet(FLOWERS_TRAIN)
)
(
    test
    .select("features_vectorized", "labelIndex")
    .write.mode("overwrite").parquet(FLOWERS_TEST)
)

N_CLASSES      = int(features_df_dense.select("labelIndex").distinct().count())
FEATURE_DIM    = 2048
FLOWER_EPOCHS  = 20
FLOWER_BATCH   = 32
FLOWER_LR      = 1e-3

print(f"Classes: {N_CLASSES}, feature dim: {FEATURE_DIM}")
print(f"Train rows: {train.count()}, test rows: {test.count()}")

Classes: 5, feature dim: 2048
Train rows: 252, test rows: 107


In [15]:
class FlowerHead(nn.Module):
        """Two-layer classification head on top of frozen ResNet-50 features."""
        def __init__(self, in_dim, n_classes, dropout=0.3):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, 256),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(256, n_classes),
            )

        def forward(self, x):
            return self.net(x)

In [16]:
FLOWERS_TRAIN_PATH = FLOWERS_TRAIN  # paths already set in cell 28
FLOWERS_TEST_PATH  = FLOWERS_TEST


def train_flower_head():
    # All imports and class definitions must live inside the function
    # (same TorchDistributor worker-isolation requirement as train_mnist).
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torch.optim as optim
    import pyarrow.parquet as pq
    from torch.utils.data import TensorDataset, DataLoader

    def load(path):
        """Read a Parquet file written by Spark and return a TensorDataset.

        Spark writes VectorUDT columns as structs {type, size, indices, values};
        to_pylist() therefore yields dicts — extract the 'values' key for the
        raw float list.
        """
        table = pq.read_table(path)
        X = torch.tensor(
            [row["values"] for row in table["features_vectorized"].to_pylist()],
            dtype=torch.float32,
        )
        y = torch.tensor(table["labelIndex"].to_pylist(), dtype=torch.long)
        return TensorDataset(X, y)

    device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_loader = DataLoader(load(FLOWERS_TRAIN_PATH), batch_size=FLOWER_BATCH, shuffle=True)
    test_loader  = DataLoader(load(FLOWERS_TEST_PATH),  batch_size=FLOWER_BATCH)

    model = FlowerHead(FEATURE_DIM, N_CLASSES).to(device)
    opt   = optim.Adam(model.parameters(), lr=FLOWER_LR)

    for epoch in range(1, FLOWER_EPOCHS + 1):
        model.train()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            F.cross_entropy(model(X), y).backward()
            opt.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            correct += (model(X).argmax(1) == y).sum().item()
            total   += y.size(0)
    print(f"FlowerHead test accuracy: {correct / total:.4f}")

    return model.state_dict()


flower_state_dict = TorchDistributor(
    num_processes=1, local_mode=True, use_gpu=False,
).run(train_flower_head)

flower_model = FlowerHead(FEATURE_DIM, N_CLASSES)
flower_model.load_state_dict(flower_state_dict)
flower_model.eval()

Started local training with 1 processes


FlowerHead test accuracy: 0.8411


Finished local training with 1 processes


FlowerHead(
  (net): Sequential(
    (0): Linear(in_features=2048, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=5, bias=True)
  )
)

The logistic regression and the PyTorch head are trained on the same features extracted by the same ResNet. Any accuracy difference is therefore purely due to the classifier — not the feature extractor. The PyTorch head has more capacity (non-linear activations, dropout regularisation) but also more hyperparameters to tune. On a dataset this small, the two are often competitive; the head's advantage grows with dataset size.

---

## Shakespeare Sentiment Analysis

We classify every Shakespeare line as positive or negative using a pre-trained DistilBERT model via HuggingFace Transformers. Inference runs inside Spark using `predict_batch_udf`, which loads the model once per worker and processes rows in batches — the same pattern used for ResNet feature extraction earlier in the notebook.

In [3]:
from pyspark.sql.functions import avg, count, stddev, rand, expr, col, length
from pyspark.sql.types import FloatType, StringType
import pandas as pd

### Load and preprocess data

We read the CSV, keep only the three columns we need, and drop rows with missing lines. We also filter lines that are too short to convey a meaningful sentiment.

In [4]:
df = spark.read.csv("shakespeare_data.csv", header=True, inferSchema=True)
df = df \
    .select("Play", "Player", "PlayerLine") \
    .na.drop(subset=["PlayerLine"]) \
    .filter(length(col("PlayerLine")) > 30)

### Build the sentiment UDF with predict_batch_udf

We use `predict_batch_udf` (the same API as the ResNet section) to run `distilbert-base-uncased-finetuned-sst-2-english` across all partitions. The model is loaded once per worker at startup; the inner function receives a NumPy array of strings and returns two arrays — the predicted label and its confidence score.

### Apply the UDF and extract sentiment class and confidence

The UDF returns a struct with two fields. We unpack them into separate columns so the rest of the pipeline can treat them as plain floats and strings.

In [9]:
from pyspark.ml.functions import predict_batch_udf
from pyspark.sql.types import StructType, StructField, StringType, FloatType

# This part of the script relies on HuggingFace. If you have a HuggingFace account (which is not necessary
# to run the code), set an HF_TOKEN environment variable to accelerate the code. 

def make_sentiment_fn():
    """Called once per worker: loads and caches the DistilBERT sentiment model."""
    from transformers import pipeline
    import numpy as np

    classifier = pipeline(
        "sentiment-analysis",
        model="distilbert-base-uncased-finetuned-sst-2-english",
        truncation=True,
        max_length=512,
    )

    def predict(texts: np.ndarray) -> dict:
        results = classifier(texts.tolist())
        labels  = np.array([r["label"].lower() for r in results])
        scores  = np.array([r["score"]          for r in results], dtype=np.float32)
        return {"sentiment_class": labels, "confidence": scores}

    return predict


sentiment_udf = predict_batch_udf(
    make_sentiment_fn,
    return_type=StructType([
        StructField("sentiment_class", StringType()),
        StructField("confidence",      FloatType()),
    ]),
    batch_size=256,
)

# Cache df and force materialization
# You may sample the dataframe (as done below) the accelerate the computation.
df = df.sample(fraction=0.25).cache()
df.count()

raw = df.repartition(4).withColumn("sentiment", sentiment_udf(col("PlayerLine")))
result = (
    raw
    .withColumn("sentiment_class", col("sentiment.sentiment_class"))
    .withColumn("confidence",      col("sentiment.confidence"))
    .drop("sentiment")
)

### Map classes to a numeric sentiment score

DistilBERT outputs binary labels (`positive` / `negative`). We map these to +1 / −1 so we can compute means and confidence intervals.

In [10]:
from pyspark.sql.functions import when

# Binary mapping: positive → +1, negative → -1
result = result.withColumn(
    "sentiment_score",
    when(col("sentiment_class") == "positive",  1.0)
    .otherwise(-1.0)
    .cast(FloatType())
)


### Statistical analysis per play

We compute mean, standard deviation, and a 95 % confidence interval for each play's sentiment score.

In [11]:
result.show(5)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 4926.28it/s]


+--------------------+--------------+--------------------+---------------+----------+---------------+
|                Play|        Player|          PlayerLine|sentiment_class|confidence|sentiment_score|
+--------------------+--------------+--------------------+---------------+----------+---------------+
|          Coriolanus|        BRUTUS|Be rein'd again t...|       positive| 0.8121626|            1.0|
|           King John|     CONSTANCE|But now will cank...|       negative| 0.8127644|           -1.0|
|Antony and Cleopatra|Second Servant|Why, this is to h...|       positive|0.99663264|            1.0|
|     Henry VI Part 2| KING HENRY VI|Re-enter WARWICK ...|       positive| 0.9556705|            1.0|
|     Henry VI Part 3|      CLARENCE|Against his broth...|       negative| 0.9883837|           -1.0|
+--------------------+--------------+--------------------+---------------+----------+---------------+
only showing top 5 rows


In [12]:
play_stats = (
    result 
    .groupBy("Play")
    .agg(
        avg("sentiment_score").alias("avg_sentiment"),
        stddev("sentiment_score").alias("sentiment_stddev"),
        count("PlayerLine").alias("line_count"),
        avg("confidence").alias("avg_confidence"),
    )
    .withColumn("ci_lower", expr("avg_sentiment - 1.96 * sentiment_stddev / SQRT(line_count)"))
    .withColumn("ci_upper", expr("avg_sentiment + 1.96 * sentiment_stddev / SQRT(line_count)"))
    .orderBy("avg_sentiment", ascending=False)
)

print("Most positive plays:")
play_stats.show(5)

print("Most negative plays:")
play_stats.orderBy("avg_sentiment", ascending=True).show(5)


Most positive plays:


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 17157.32it/s]


+--------------------+-------------------+------------------+----------+------------------+--------------------+-------------------+
|                Play|      avg_sentiment|  sentiment_stddev|line_count|    avg_confidence|            ci_lower|           ci_upper|
+--------------------+-------------------+------------------+----------+------------------+--------------------+-------------------+
|            Pericles|  0.226890756302521|0.9780382889970813|       119|0.9560335033080157|  0.0511637753088261| 0.4026177372962159|
| Taming of the Shrew|0.19298245614035087|0.9840838646332836|       171|0.9535526988799112| 0.04548309910360232| 0.3404818131770994|
|A Midsummer night...|0.18181818181818182|0.9878325655940079|       110|0.9586321267214688|-0.00278665174536...|0.36642301538172817|
|          Henry VIII|0.15976331360946747|0.9900889526311704|       169|0.9475460137135884|0.010488363828152547| 0.3090382633907824|
|             Henry V| 0.1368421052631579|0.9932100308663488|       1

+------------------+--------------------+------------------+----------+------------------+--------------------+--------------------+
|              Play|       avg_sentiment|  sentiment_stddev|line_count|    avg_confidence|            ci_lower|            ci_upper|
+------------------+--------------------+------------------+----------+------------------+--------------------+--------------------+
|       Richard III|-0.21238938053097345| 0.979354231078403|       226|0.9553760340255973| -0.3400749023620428| -0.0847038586999041|
|           Othello|-0.20245398773006135|0.9823096316213006|       163|0.9657554191314369|-0.35325722741193744|-0.05165074804818526|
|         King Lear| -0.1377245508982036|0.9934494345926307|       167|0.9493851790171184|-0.28840038445399685|0.012951282657589686|
|A Comedy of Errors|-0.11627906976744186|0.9990419487296204|        86|0.9543832692989084|-0.32742881896848314| 0.09487067943359943|
|         Cymbeline|-0.11330049261083744|0.9960170399700232|       20

### Character-level sentiment

Which characters speak with consistently positive or negative dialogue?

In [ ]:
character_stats = (
    result.groupBy("Player")
    .agg(
        avg("sentiment_score").alias("avg_sentiment"),
        count("PlayerLine").alias("line_count"),
        avg("confidence").alias("avg_confidence"),
    )
    .orderBy("avg_sentiment", ascending=False)
)

print("Most positive characters (min 20 lines):")
character_stats.filter((col("avg_sentiment") > 0) & (col("line_count") >= 20)).show(5)

print("Most negative characters (min 20 lines):")
character_stats.filter((col("avg_sentiment") < 0) & (col("line_count") >= 20)).show(5)
